In [ ]:
# Install Python libraries
! pip3 install numpy==1.23.5
! pip3 install pandas==2.0.1

In [ ]:
rscript0 = f"""################################# Install R libraries
if (!requireNamespace("BiocManager", quietly = TRUE)) {
  install.packages("BiocManager")
}

if (!requireNamespace("remotes", quietly = TRUE)) {
  install.packages("remotes")
}

## =========================
## 1. CRAN packages
## =========================
install.packages(c(
  "ggplot2",
  "dplyr",
  "future",
  "patchwork",
  "stringr",
  "Seurat"
))

## =========================
## 2. Bioconductor packages
## =========================
BiocManager::install(c(
  "GenomicRanges",
  "GenomeInfoDb",
  "EnsDb.Mmusculus.v79",
  "BSgenome.Mmusculus.UCSC.mm10"
), ask = FALSE, update = FALSE)

## =========================
## 3. Signac
## =========================
install.packages("Signac")

## =========================
## 4. monocle3
## =========================
if (!requireNamespace("monocle3", quietly = TRUE)) {
  remotes::install_github("cole-trapnell-lab/monocle3")
}

## =========================
## 5. SeuratWrappers
## =========================
if (!requireNamespace("SeuratWrappers", quietly = TRUE)) {
  remotes::install_github("satijalab/seurat-wrappers")
}"""

f = open(f'install_libraries.r', 'w')
f.write(rscript0)
f.close()

!Rscript install_libraries.r

# 0. Library and settings

In [18]:
# Library
import os
import pickle
import math
import numpy as np
import pandas as pd

In [26]:
# Make directories
os.makedirs('Fig', exist_ok=True)
os.makedirs('Fig/Supplementary_Fig3', exist_ok=True)

In [20]:
# Figure number definitions
Fig1e = f'Fig/Figure1(e)'
Fig1f = f'Fig/Figure1(f)'
Supplementary_Fig3 = f'Fig/Supplementary_Figure2'

# 1. Cluster definition file for Link Plot

In [15]:
# Loading cluster definition
df_cdef = pd.read_csv(f'cluster_def_by_monocle_final.v2.1.rev.csv', index_col=0)

# Rename cluster numbers
strings = ['a' if b[-1] == '1' else 'b' if b[-1] == '2' else 'c' for b in df_cdef.index.tolist()]
cnums = df_cdef['monocle_clusters'].tolist()
df_cdef['monocle_clusters'] = [f'{cn}{s}'for cn, s in zip(cnums, strings)]

# Rename barcodes
df_cdef.index = [f'pln_{b[:-2]}' if b[-1] == '1' else f'mln_{b[:-2]}' if b[-1] == '2' else f'pp_{b[:-2]}'for b in df_cdef.index.tolist()]

# Save
df_cdef.to_csv(f'cluster_def_by_monocle_final.v2.1.rev.link_plot.csv')

# 2. Calculate Gene Activity

In [27]:
rscript1 = f"""################################
# This is an R script.
# --------------------------------------------------------
# Filename: calc_gene_activity.r
# --------------------------------------------------------
# How to run
# $ Rscript calc_gene_activity.r
# --------------------------------------------------------
# outputs
# - Signac/PLN/pln.atac.v2.ga.RData
# - Signac/atac.v2.ga.RData
################################
# Library
library(monocle3)
library(ggplot2)
library(dplyr)
library(SeuratWrappers)
library(stringr)
library(Signac)
library(Seurat)
library(GenomicRanges)
library(future)
library(GenomeInfoDb)
library(EnsDb.Mmusculus.v79)
library(BSgenome.Mmusculus.UCSC.mm10)
library(patchwork)
set.seed(1234)

# Memory
options(future.globals.maxSize = 16000 * 1024^2)

# Loading data
load("Signac/PLN/pln.atac.v2.RData") # pln
load("Signac/atac.v2.RData") # combined

# Cluster definition
cdef1 <- read.csv("cluster_def_by_monocle_final.v2.1.rev.link_plot.csv", row.names=1)

# Cluster names definition
tmp <- cdef1[rownames(combined@meta.data),]
tmp[is.na(tmp)] <- "0"
combined@meta.data["cdef1"] = tmp
Idents(combined) <- "cdef1"
combined <- RenameIdents(combined, '0' = 'Unknown')
combined <- RenameIdents(combined, '1a' = 'Art (PLN)')
combined <- RenameIdents(combined, '3a' = "CapEC2' (PLN)")
combined <- RenameIdents(combined, '4a' = 'CapEC2 (PLN)')
combined <- RenameIdents(combined, '5a' = 'CapEC1 (PLN)')
combined <- RenameIdents(combined, '6a' = 'CRP (PLN)')
combined <- RenameIdents(combined, '7a' = 'TrEC (PLN)')
combined <- RenameIdents(combined, '8a' = 'HEC (PLN)')
combined <- RenameIdents(combined, '9a' = 'Vn (PLN)')
combined <- RenameIdents(combined, '1b' = 'Art (MLN)')
combined <- RenameIdents(combined, '3b' = "CapEC2' (MLN)")
combined <- RenameIdents(combined, '4b' = 'CapEC2 (MLN)')
combined <- RenameIdents(combined, '5b' = 'CapEC1 (MLN)')
combined <- RenameIdents(combined, '6b' = 'CRP (MLN)')
combined <- RenameIdents(combined, '7b' = 'TrEC (MLN)')
combined <- RenameIdents(combined, '8b' = 'HEC (MLN)')
combined <- RenameIdents(combined, '9b' = 'Vn (MLN)')
combined <- RenameIdents(combined, '1c' = 'Art (PP)')
combined <- RenameIdents(combined, '3c' = "CapEC2' (PP)")
combined <- RenameIdents(combined, '4c' = 'CapEC2 (PP)')
combined <- RenameIdents(combined, '5c' = 'CapEC1 (PP)')
combined <- RenameIdents(combined, '6c' = 'CRP (PP)')
combined <- RenameIdents(combined, '7c' = 'TrEC (PP)')
combined <- RenameIdents(combined, '8c' = 'HEC (PP)')
combined <- RenameIdents(combined, '9c' = 'Vn (PP)')

# Add scRNA-Seq data (Gene activity)
# PLN
gene.activities.pln <- GeneActivity(pln)
pln[['RNA']] <- CreateAssayObject(counts = gene.activities.pln)
pln <- NormalizeData(
  object = pln,
  assay = 'RNA',
  normalization.method = 'LogNormalize',
  scale.factor = median(pln$nCount_RNA)
)
# ALL
gene.activities <- GeneActivity(combined)
combined[['RNA']] <- CreateAssayObject(counts = gene.activities)
combined <- NormalizeData(
  object = combined,
  assay = 'RNA',
  normalization.method = 'LogNormalize',
  scale.factor = median(pln$nCount_RNA)
)

# Save
save(combined, file="Signac/PLN/pln.atac.v2.ga.RData")
save(combined, file="Signac/atac.v2.ga.RData")
"""

f = open('calc_gene_activity.r', 'w')
f.write(rscript1)
f.close()

In [ ]:
!Rscript calc_gene_activity.r

# 3. Link plot

In [28]:
rscript2 = f"""################################
# This is an R script.
# --------------------------------------------------------
# Filename: link_plot.r
# --------------------------------------------------------
# How to run
# $ Rscript link_plot.r
# --------------------------------------------------------
# outputs
# - Fig/Fig1(e).pdf
# - Fig/Fig1(f).pdf
# - Fig/Supplementary_Fig3/*
################################
# Library
library(monocle3)
library(ggplot2)
library(dplyr)
library(SeuratWrappers)
library(stringr)
library(Signac)
library(Seurat)
library(GenomicRanges)
library(future)
library(GenomeInfoDb)
library(EnsDb.Mmusculus.v79)
library(BSgenome.Mmusculus.UCSC.mm10)
library(patchwork)
set.seed(1234)

# Memory
options(future.globals.maxSize = 16000 * 1024^2)

# Loading data
load("Signac/atac.v2.ga.RData")

# Link Plot (function)
plotCombined <- function(
    obj,
    gene,
    region,
    fn,
    upstream=1000,
    downstream=1000,
    cdef="cdef1"
){{
    # assay
    DefaultAssay(obj) <- "peaks"
    
    # first compute the GC content for each peak
    obj <- RegionStats(obj, genome = BSgenome.Mmusculus.UCSC.mm10)
    
    # link peaks to genes
    obj <- LinkPeaks(
      object = obj,
      peak.assay = "peaks",
      expression.assay = "RNA",
      genes.use = c(gene)
    )
    
    # set plotting order
    levels(obj) <- c(
        "Art (PLN)",
        "Art (MLN)",
        "Art (PP)",
        "CapEC2' (PLN)",
        "CapEC2' (MLN)",
        "CapEC2' (PP)",
        "CapEC2 (PLN)",
        "CapEC2 (MLN)",
        "CapEC2 (PP)",
        "CapEC1 (PLN)",
        "CapEC1 (MLN)",
        "CapEC1 (PP)",
        "CRP (PLN)",
        "CRP (MLN)",
        "CRP (PP)",
        "TrEC (PLN)",
        "TrEC (MLN)",
        "TrEC (PP)",
        "HEC (PLN)",
        "HEC (MLN)",
        "HEC (PP)",
        "Vn (PLN)",
        "Vn (MLN)",
        "Vn (PP)"
    )

    # set color
    colors = c(
        "#D0021B",
        "#D0021B",
        "#D0021B",
        "#BA98FF",
        "#BA98FF",
        "#BA98FF",
        "#FF789B",
        "#FF789B",
        "#FF789B",
        "#50E3C2",
        "#50E3C2",
        "#50E3C2",
        "#7ED321",
        "#7ED321",
        "#7ED321",
        "#4A90E2",
        "#4A90E2",
        "#4A90E2",
        "#F5A623",
        "#F5A623",
        "#F5A623",
        "#9013FE",
        "#9013FE",
        "#9013FE"
    )

    cov_plot <- CoveragePlot(
      object = obj,
      region = region,
      extend.upstream = upstream,
      extend.downstream = downstream,
      annotation = FALSE,
      peaks = FALSE,
      links = FALSE
    ) + scale_fill_manual(values=colors)
    
    gene_plot <- AnnotationPlot(
      object = obj,
      region = region,
      extend.upstream = upstream,
      extend.downstream = downstream
    )
    
    peak_plot <- PeakPlot(
      object = obj,
      region = region,
      extend.upstream = upstream,
      extend.downstream = downstream
    )
    
    link_plot <- LinkPlot(
      object = obj,
      region = region,
      extend.upstream = upstream,
      extend.downstream = downstream
    )
    
    expr_plot <- ExpressionPlot(
      object = obj,
      features = gene,
      assay = "RNA"
    ) + scale_fill_manual(values=colors)
    
    p <- CombineTracks(
      plotlist = list(cov_plot, peak_plot, gene_plot, link_plot),
      expression.plot = expr_plot,
      heights = c(10, 1, 2, 3),
      widths = c(10, 2)
    )
    
    ggsave(filename = fn, plot = p)
}}

plotCombined(
    combined,
    "Chst4",
    "chr8-110029075-110039334",
    "Fig/Fig1(e).pdf",
    upstream = 10000,
    downstream = 10000,
    cdef = "cdef1"
)

plotCombined(
    combined,
    "Madcam1",
    "chr10-79664559-79668542",
    "Fig/Fig1(f).pdf",
    upstream = 10000,
    downstream = 10000,
    cdef = "cdef1"
)

plotCombined(
    combined,
    "Fut7",
    "chr2-25423242-25426374",
    "Fig/Supplementary_Fig3/Supplementary_Fig3_Fut7.pdf",
    upstream = 10000,
    downstream = 10000,
    cdef = "cdef1"
)

plotCombined(
    combined,
    "Gcnt1",
    "chr19-17326141-17356667",
    "Fig/Supplementary_Fig3/Supplementary_Fig3_Gcnt1.pdf",
    upstream = 10000,
    downstream = 10000,
    cdef = "cdef1"
)

plotCombined(
    combined,
    "Glycam1",
    "chr15-103562759-103565081",
    "Fig/Supplementary_Fig3/Supplementary_Fig3_Glycam1.pdf",
    upstream = 10000,
    downstream = 10000,
    cdef = "cdef1"
)
"""

f = open(f'link_plot.r', 'w')
f.write(rscript2)
f.close()

In [ ]:
!Rscript link_plot.r